#**Project Overview: Content Review Quality Optimization**

###**Phase 1**
**Synthetic Data Generation**


___________________________________


**Install Packages**


In [3]:
!pip install pandas

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# --- 1. Define Dataset Parameters ---
N_ROWS = 20000
START_DATE = datetime(2025, 8, 1)
END_DATE = datetime(2025, 8, 31)

# Reviewer/Team Data
REGIONS = ['North America (NA)', 'Europe (EU)', 'Asia-Pacific (APAC)', 'Latin America (LATAM)']
TEAM_LEVELS = ['L1 - General', 'L2 - Complex', 'L3 - Policy Expert']

# Content Data
POLICY_AREAS = ['Hate Speech', 'Violent Content', 'Spam & Scams', 'Intellectual Property']
CONTENT_TYPES = ['Video', 'Image', 'Text', 'Livestream']

# --- 2. Generate Synthetic Data with Corrected Logic ---

# Generate Date/Time
time_delta = END_DATE - START_DATE
dates = [START_DATE + timedelta(seconds=np.random.randint(0, time_delta.total_seconds())) for _ in range(N_ROWS)]

# Generate Reviewer/Team Data
reviewer_id = [f'R{i:05d}' for i in range(N_ROWS)]
region = np.random.choice(REGIONS, N_ROWS, p=[0.35, 0.30, 0.25, 0.10])
team_level = np.random.choice(TEAM_LEVELS, N_ROWS, p=[0.55, 0.35, 0.10])

# Generate Content & Review Metrics
policy_area = np.random.choice(POLICY_AREAS, N_ROWS, p=[0.40, 0.25, 0.20, 0.15])
content_type = np.random.choice(CONTENT_TYPES, N_ROWS, p=[0.30, 0.35, 0.25, 0.10])

# Content Complexity Score (1 to 10) - influenced by Policy Area and Content Type
complexity = np.random.normal(loc=5.5, scale=2.0, size=N_ROWS).astype(int)
complexity = np.clip(complexity, 1, 10)

# Decision Accuracy (The core quality metric: Did the reviewer's decision match the expert consensus?)
# Correction: Now accuracy is explicitly tied to both team level and complexity, creating a measurable correlation.
accuracy_base = 0.90
accuracy = np.where(team_level == 'L3 - Policy Expert', np.random.normal(accuracy_base + 0.05, 0.03, N_ROWS),
                    np.where(team_level == 'L2 - Complex', np.random.normal(accuracy_base, 0.04, N_ROWS),
                             np.random.normal(accuracy_base - 0.05, 0.05, N_ROWS)))
# Add a negative correlation with complexity
accuracy = accuracy - (complexity / 1000) * np.random.normal(5, 1, N_ROWS)
accuracy = np.clip(accuracy, 0.75, 0.99).round(4)

# Review Time (in seconds)
# Correction: The AI Triage impact is now more significant.
time_base = 120
review_time = (time_base + (complexity * 15) + np.random.normal(0, 30, N_ROWS))
review_time = np.where(team_level == 'L3 - Policy Expert', review_time * 1.5,
                      np.where(team_level == 'L2 - Complex', review_time * 1.2, review_time))

ai_triage_used = np.random.choice([True, False], N_ROWS, p=[0.60, 0.40])
review_time = np.where(ai_triage_used, review_time * 0.8, review_time) # 20% reduction with AI
review_time = np.clip(review_time, 10, 600).astype(int)

# Volume (Total items reviewed by a reviewer in the month)
volume = np.random.lognormal(mean=7.5, sigma=0.8, size=N_ROWS).astype(int)

# --- 3. Create DataFrame ---
data = pd.DataFrame({
    'review_id': [f'REV{i:06d}' for i in range(N_ROWS)],
    'review_date_time': dates,
    'reviewer_id': reviewer_id,
    'region': region,
    'team_level': team_level,
    'policy_area': policy_area,
    'content_type': content_type,
    'content_complexity_score': complexity,
    'ai_triage_used': ai_triage_used,
    'review_time_seconds': review_time,
    'decision_accuracy_rate': accuracy,
    'reviewer_monthly_volume': volume
})

# --- 4. Derive Operational Metrics ---
# Quality: The 1 - accuracy rate is the "Error Rate"
data['decision_error_rate'] = 1 - data['decision_accuracy_rate']
# Efficiency: Reviews per Minute
data['reviews_per_minute'] = 60 / data['review_time_seconds']

# --- 5. Save Data ---
FILE_NAME = 'content_review_data.csv'
data.to_csv(FILE_NAME, index=False)

print(f"✅ Data Generation Complete.")
print(f"Shape of the generated data: {data.shape}")
print(f"File saved as: {FILE_NAME}")

# Display the first few rows and data types for verification
print("\nFirst 5 Rows:")
print(data.head())
print("\nData Types:")
print(data.dtypes)

✅ Data Generation Complete.
Shape of the generated data: (20000, 14)
File saved as: content_review_data.csv

First 5 Rows:
   review_id    review_date_time reviewer_id                 region  \
0  REV000000 2025-08-26 16:25:10      R00000     North America (NA)   
1  REV000001 2025-08-26 19:11:24      R00001  Latin America (LATAM)   
2  REV000002 2025-08-28 06:32:10      R00002    Asia-Pacific (APAC)   
3  REV000003 2025-08-20 14:12:23      R00003            Europe (EU)   
4  REV000004 2025-08-02 06:37:48      R00004            Europe (EU)   

           team_level            policy_area content_type  \
0  L3 - Policy Expert            Hate Speech   Livestream   
1        L2 - Complex  Intellectual Property        Image   
2        L2 - Complex        Violent Content        Video   
3        L2 - Complex        Violent Content         Text   
4        L1 - General           Spam & Scams         Text   

   content_complexity_score  ai_triage_used  review_time_seconds  \
0              